# BioAI Hub — Layer 3: Métricas reales y augmentación

En Layer 2 entrené BioAICNN y obtuve accuracy en test. En esta layer evalúo el modelo con métricas más adecuadas para clasificación médica (AUC-ROC, precisión, recall) y entreno una segunda versión con augmentación más agresiva para comparar resultados.

## 0. Setup

Mismo bloque de inicio que en layers anteriores. Si el runtime se reinició, esto restaura todo.

In [ ]:
import os
import torch
import torch.nn as nn
import numpy as np

WORK_DIR  = os.path.expanduser('~/bioai-colab-data')
DATA_DIR  = os.path.join(WORK_DIR, 'chest-xray-data')
CHEST_DIR = os.path.join(DATA_DIR, 'chest_xray')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
import os

if not os.path.exists(CHEST_DIR):
    print('Dataset no encontrado.')
    print(f'Corré Layer 2 primero para descargar el dataset en: {CHEST_DIR}')
else:
    print('Dataset disponible')

In [ ]:
class BioAICNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 28 * 28, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

print("Arquitectura definida")

## 1. Por qué accuracy no alcanza

El dataset tiene ~1300 imágenes NORMAL y ~3900 PNEUMONIA en training. Si el modelo aprendiera a predecir siempre PNEUMONIA tendría ~75% de accuracy sin haber aprendido nada útil.

En medicina esto es especialmente crítico porque los errores no tienen el mismo costo:
- **Falso negativo** (decir NORMAL cuando hay neumonía): el paciente no recibe tratamiento
- **Falso positivo** (decir PNEUMONIA cuando está sano): tratamiento innecesario

Las métricas que importan:
- **Recall (Sensibilidad)**: de todos los casos reales de neumonía, ¿qué porcentaje detecté? Quiero esto alto.
- **Precisión**: de todos los que dije que tenían neumonía, ¿cuántos realmente la tenían?
- **AUC-ROC**: qué tan bien separa el modelo las dos clases en general, independientemente del umbral de decisión. 0.5 = azar, 1.0 = perfecto.

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import platform

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
NUM_WORKERS   = 0 if platform.system() == 'Windows' else 4

transform_eval = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

test_ds     = datasets.ImageFolder(os.path.join(CHEST_DIR, 'test'), transform=transform_eval)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=NUM_WORKERS)

model_v1   = BioAICNN().to(device)
checkpoint = torch.load(os.path.join(WORK_DIR, 'bioaicnn_best.pth'), map_location=device, weights_only=True)
model_v1.load_state_dict(checkpoint['model_state_dict'])
print(f"Modelo v1 cargado — epoch {checkpoint['epoch']}, val_loss={checkpoint['val_loss']:.4f}")
print(f'Clases: {test_ds.classes}')

In [ ]:
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve
)
import matplotlib.pyplot as plt
import seaborn as sns

def evaluar(model, loader):
    model.eval()
    all_labels, all_preds, all_probs = [], [], []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)[:, 1]  # prob de clase PNEUMONIA
            preds = outputs.argmax(1)

            all_labels.extend(labels.numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    return np.array(all_labels), np.array(all_preds), np.array(all_probs)

labels_v1, preds_v1, probs_v1 = evaluar(model_v1, test_loader)

auc_v1 = roc_auc_score(labels_v1, probs_v1)
print(f"AUC-ROC v1: {auc_v1:.4f}")
print()
print(classification_report(labels_v1, preds_v1, target_names=test_ds.classes))

In [ ]:
fpr, tpr, _ = roc_curve(labels_v1, probs_v1)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f'BioAICNN v1 (AUC = {auc_v1:.3f})')
plt.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Azar')
plt.xlabel('Tasa de falsos positivos')
plt.ylabel('Tasa de verdaderos positivos (Recall)')
plt.title('Curva ROC — test set')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(WORK_DIR, 'roc_v1.png'), dpi=120, bbox_inches='tight')
plt.show()

## 2. Augmentación más agresiva

En Layer 2 usé solo flip horizontal y rotación leve. Ahora agrego:

- `RandomAffine` con traslación: las radiografías no siempre están perfectamente centradas
- `ColorJitter` con brillo y contraste: simula variaciones entre equipos de rayos X
- `RandomErasing`: tapa zonas aleatorias de la imagen — obliga al modelo a no depender de una región específica

La idea es que el modelo vea más variedad artificial durante el entrenamiento y generalice mejor.

In [ ]:
from pathlib import Path

train_dir = Path(CHEST_DIR) / 'train'

conteo = {}
for clase in ['NORMAL', 'PNEUMONIA']:
    conteo[clase] = len(list((train_dir / clase).glob('*.jpeg')) +
                        list((train_dir / clase).glob('*.jpg')) +
                        list((train_dir / clase).glob('*.png')))

n_total = sum(conteo.values())
class_weights = torch.tensor([
    n_total / (2 * conteo['NORMAL']),
    n_total / (2 * conteo['PNEUMONIA'])
]).to(device)

transform_train_v2 = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.1)),
])

train_ds_v2 = datasets.ImageFolder(str(train_dir),                          transform=transform_train_v2)
val_ds_v2   = datasets.ImageFolder(str(Path(CHEST_DIR) / 'val'),             transform=transform_eval)

train_loader_v2 = DataLoader(train_ds_v2, batch_size=32, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
val_loader_v2   = DataLoader(val_ds_v2,   batch_size=32, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print('DataLoaders v2 listos')

In [ ]:
# Ver cómo afecta la augmentación a una misma imagen
from torchvision.utils import make_grid
import matplotlib.image as mpimg
from PIL import Image

sample_path = list((train_dir / 'PNEUMONIA').glob('*.jpeg'))[0]
sample_img  = Image.open(sample_path).convert('RGB')

augmented = [transform_train_v2(sample_img) for _ in range(6)]

# Desnormalizar para visualizar
mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
std  = torch.tensor(IMAGENET_STD).view(3, 1, 1)
augmented_vis = [(t * std + mean).clamp(0, 1) for t in augmented]

grid = make_grid(augmented_vis, nrow=3)
plt.figure(figsize=(12, 8))
plt.imshow(grid.permute(1, 2, 0).numpy())
plt.title('6 versiones augmentadas de la misma radiografía')
plt.axis('off')
plt.tight_layout()
plt.show()

## 3. Entrenamiento v2

Mismo proceso que en Layer 2 pero con el nuevo transform. Agrego un learning rate scheduler — reduce el lr a la mitad si la val_loss no mejora en 3 epochs consecutivas. Ayuda a escapar de mínimos locales en las últimas epochs.

In [ ]:
from tqdm import tqdm

model_v2  = BioAICNN().to(device)
optimizer = torch.optim.Adam(model_v2.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss(weight=class_weights)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3
)

EPOCHS = 10
CHECKPOINT_V2 = os.path.join(WORK_DIR, 'bioaicnn_v2_best.pth')

history_v2 = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_loss = float('inf')


def run_epoch(model, loader, training=True):
    model.train() if training else model.eval()
    total_loss, correct, total = 0.0, 0, 0

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for images, labels in tqdm(loader, leave=False):
            images, labels = images.to(device), labels.to(device)
            if training:
                optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            if training:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * images.size(0)
            correct    += (outputs.argmax(1) == labels).sum().item()
            total      += images.size(0)

    return total_loss / total, correct / total


for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = run_epoch(model_v2, train_loader_v2, training=True)
    val_loss,   val_acc   = run_epoch(model_v2, val_loader_v2,   training=False)

    scheduler.step(val_loss)

    history_v2['train_loss'].append(train_loss)
    history_v2['val_loss'].append(val_loss)
    history_v2['train_acc'].append(train_acc)
    history_v2['val_acc'].append(val_acc)

    mejoro = val_loss < best_val_loss
    if mejoro:
        best_val_loss = val_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model_v2.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss,
            'val_acc': val_acc,
        }, CHECKPOINT_V2)

    print(f"Epoch {epoch:02d}/{EPOCHS}  "
          f"train_loss={train_loss:.4f}  train_acc={train_acc:.3f}  "
          f"val_loss={val_loss:.4f}  val_acc={val_acc:.3f}"
          f"{'  ← guardado' if mejoro else ''}")

## 4. Comparativa v1 vs v2

In [ ]:
# Cargar mejor checkpoint v2
ckpt_v2 = torch.load(CHECKPOINT_V2, map_location=device)
model_v2.load_state_dict(ckpt_v2['model_state_dict'])

labels_v2, preds_v2, probs_v2 = evaluar(model_v2, test_loader)
auc_v2 = roc_auc_score(labels_v2, probs_v2)

print(f"AUC-ROC v2: {auc_v2:.4f}")
print()
print(classification_report(labels_v2, preds_v2, target_names=test_ds.classes))

In [ ]:
from sklearn.metrics import accuracy_score

acc_v1 = accuracy_score(labels_v1, preds_v1)
acc_v2 = accuracy_score(labels_v2, preds_v2)

# Curvas ROC comparadas
fpr_v2, tpr_v2, _ = roc_curve(labels_v2, probs_v2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Curva ROC
axes[0].plot(fpr,    tpr,    label=f'v1 — aug básica  (AUC={auc_v1:.3f})')
axes[0].plot(fpr_v2, tpr_v2, label=f'v2 — aug agresiva (AUC={auc_v2:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.4)
axes[0].set_xlabel('Tasa de falsos positivos')
axes[0].set_ylabel('Recall')
axes[0].set_title('Curva ROC')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Matrices de confusión
cm_v1 = confusion_matrix(labels_v1, preds_v1)
cm_v2 = confusion_matrix(labels_v2, preds_v2)

# Mostrar solo v2 para no recargar el plot
sns.heatmap(cm_v2, annot=True, fmt='d', cmap='Blues',
            xticklabels=test_ds.classes,
            yticklabels=test_ds.classes, ax=axes[1])
axes[1].set_xlabel('Predicho')
axes[1].set_ylabel('Real')
axes[1].set_title(f'Confusión v2 (aug agresiva)')

plt.suptitle('BioAICNN — comparativa v1 vs v2', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(WORK_DIR, 'comparativa_v1_v2.png'), dpi=120, bbox_inches='tight')
plt.show()

print("\n--- Resumen ---")
print(f"{'':20s} {'v1':>10} {'v2':>10}")
print(f"{'Accuracy':20s} {acc_v1:>10.3f} {acc_v2:>10.3f}")
print(f"{'AUC-ROC':20s} {auc_v1:>10.3f} {auc_v2:>10.3f}")

## 5. Qué modelo llevar a las próximas layers

El criterio de elección es AUC-ROC sobre el test set, no accuracy. La siguiente celda elige automáticamente el mejor y lo guarda como `bioaicnn_final.pth` en Drive para usarlo como referencia comparativa en Layer 5 (transfer learning con EfficientNet).

In [ ]:
import shutil

if auc_v2 >= auc_v1:
    mejor_version = 'v2'
    src = CHECKPOINT_V2
else:
    mejor_version = 'v1'
    src = os.path.join(WORK_DIR, 'bioaicnn_best.pth')

dst = os.path.join(WORK_DIR, 'bioaicnn_final.pth')
shutil.copy(src, dst)

print(f"Mejor modelo: {mejor_version} (AUC-ROC={max(auc_v1, auc_v2):.4f})")
print(f"Guardado en: {dst}")
print()
print("Archivos en Drive:")
for f in sorted(os.listdir(WORK_DIR)):
    size = os.path.getsize(os.path.join(WORK_DIR, f)) / 1e6
    print(f"  {f:40s} {size:.1f} MB")